# Label-Agnostic Bayesian Optimization for IDS (Label Trainer)
## 1. Introduction

This notebook develops and evaluates an **Intrusion Detection System (IDS) label trainer** using a sequence of methods:

1. **Manual & automated hyperparameter optimisation (HPO)** for LightGBM classifiers.  
2. **Bayesian Optimisation (BO)** with checkpointing and resumable search for efficient black-box tuning.  
3. An **ensemble workflow** that integrates optimised models.  
4. A compact **1D-CNN baseline** trained on raw feature sequences for comparison.  
5. Unified **evaluation tools** (PR-AUC, F1@τ, learning curves, calibration, grid utilities) to justify model selection.

**PCMLAI alignment**
- **Module 3:** Probabilistic modelling (Gaussian pdf/cdf), evaluation under imbalance (Precision–Recall curves).  
- **Module 10:** Model selection & black-box optimisation (grid/random/BO, ensembles).  
- **DL note:** A small 1D-CNN baseline for contrast against classical ML.  
- **Bandits/RL link:** Explore–exploit in BO acquisition functions (multi-armed bandit intuition).

---

### Why Hyperparameter Optimisation Matters
We tune hyperparameters $\theta$ (e.g., `learning_rate`, `num_leaves`, `max_depth`) to maximise:
\[
J(\theta) = \text{PR-AUC from stratified CV}.
\]
In IDS, training is **expensive** (folds), scores **noisy** (fold variance), and the objective **non-convex**.
- **Grid search**: exhaustive but wasteful.  
- **Random search**: ignores prior evaluations.  
- **Bayesian Optimisation**: uses a surrogate + acquisition to pick *informative* trials.  
- **Ensembles**: combine complementary models for stability under class imbalance.

---

### Bayesian Optimisation: Concept
Treat $J(\theta)$ as black-box $f(\theta)$. BO iterates:  
1) evaluate initial points; 2) fit a **surrogate** (e.g., GP) for mean $\mu_t$ and uncertainty $\sigma_t$;  
3) choose next $\theta$ via an **acquisition** balancing **exploration** ($\sigma_t$) and **exploitation** ($\mu_t$);  
4) repeat until budget; return best.

**GP posterior (Module 3):**
\[
\mu_t(\theta) = k(\theta,\Theta_t)K_t^{-1}\mathbf y_t,\quad
\sigma_t^2(\theta) = k(\theta,\theta) - k(\theta,\Theta_t)K_t^{-1}k(\Theta_t,\theta).
\]

**Acquisitions (Module 10, Bandits):**
\[
EI(\theta) = (\mu_t-f^*-\xi)\,\Phi(z) + \sigma_t\,\phi(z),\;
PI(\theta)=\Phi\!\left(\frac{\mu_t-f^*-\xi}{\sigma_t}\right),\;
UCB(\theta)=\mu_t + \kappa\,\sigma_t.
\]
$\xi\!\ge\!0$ and $\kappa\!>\!0$ control **explore–exploit** (bandit trade-off).

---

### Notebook Flow
1) Data prep → 2) Baseline → 3) HPO (Grid/Random/BO/Enhanced) → 4) CNN baseline →  
5) Ensembles → 6) Analysis plots → 7) Champion + Auto-wire docs.


## 2. Configuration, Utilities, and Staging
- Set random/threads, define staging helpers (`stage_dir`, `save_stage`, `load_stage`).
- Provide metric utilities used throughout (AUPRC, F1@τ, TPR/TNR).


In [1]:
# Reproducibility & system threads
import os, json, time, math, joblib, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score, f1_score, precision_recall_curve, confusion_matrix
)
from sklearn.model_selection import train_test_split, StratifiedKFold

RANDOM_STATE = 42
N_THREADS = max(1, (os.cpu_count() or 4) - 1)
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(N_THREADS))

# Data path (change to your CSV if needed)
DATA_PATH = "archive/Payload_data_UNSW.csv"

# --- Staging helpers (resume-safe) ---
STAGING = Path("staging"); STAGING.mkdir(parents=True, exist_ok=True)

def stage_dir(name: str) -> Path:
    """Create/return a stage subfolder under ./staging for artifacts."""
    d = STAGING / name
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_stage(name: str, manifest: dict, results: pd.DataFrame | None = None, model=None):
    """Persist manifest + optional results table + optional model for reproducibility."""
    d = stage_dir(name)
    (d / "manifest.json").write_text(json.dumps(manifest, indent=2))
    if isinstance(results, pd.DataFrame):
        (d / "results.csv").write_text(results.to_csv(index=False))
    if model is not None:
        try:
            joblib.dump(model, d / "model.joblib")
        except Exception as e:
            print("Warning: model save failed:", e)

def load_stage(name: str) -> dict | None:
    """Read manifest.json if present (returns dict or None)."""
    p = stage_dir(name) / "manifest.json"
    if p.exists():
        try:
            return json.loads(p.read_text())
        except Exception:
            return None
    return None

def now_utc() -> str:
    """ISO8601 UTC timestamp string."""
    return datetime.utcnow().isoformat() + "Z"

# --- Metric helpers (used notebook-wide) ---
def sweep_f1(y_true, p_hat, grid=np.linspace(0.05, 0.95, 19)):
    """Return (best_F1, tau) by sweeping probability thresholds on a grid."""
    f1s = [(t, f1_score(y_true, (p_hat >= t).astype(int), zero_division=0)) for t in grid]
    t, f = max(f1s, key=lambda z: z[1])
    return float(f), float(t)

def tpr_tnr_at_tau(y_true, p_hat, tau: float):
    """Compute TPR/TNR at decision threshold tau."""
    yb = (p_hat >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yb).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    tnr = tn / (tn + fp) if (tn + fp) else 0.0
    return float(tpr), float(tnr)


In [2]:
# 2) Load data & robust binary label mapping
DF = pd.read_csv(DATA_PATH)

# detect a label column
label_candidates = ["label","label_str","attack_cat","class","target","y"]
lbl_col = next((c for c in label_candidates if c in DF.columns), None)
assert lbl_col is not None, f"No label column found in {DATA_PATH}. Have: {list(DF.columns)[:30]}"

s = DF[lbl_col]
benign_tokens = {"benign","normal","background","bg","clean"}

if pd.api.types.is_numeric_dtype(s) and set(pd.unique(s.dropna())) <= {0,1}:
    y = s.astype(int).values
else:
    y = (~s.astype("string").str.strip().str.lower().isin(benign_tokens)).astype(int).values

DF = DF.copy()
DF["label"] = y
X_full = DF.drop(columns=["label"])

print("Dataset:", DF.shape, "| Pos rate:", float(np.mean(y)))


Dataset: (79881, 1505) | Pos rate: 0.7371089495624742


## 3. Data Loading and Robust Label Mapping
- Read CSV, detect label column.
- Convert label to **binary**: *benign → 0*, *non-benign → 1*.  
- Keep features as-is; categories remain categorical for LightGBM.


In [3]:
# Load dataset
DF = pd.read_csv(DATA_PATH)

# Detect a label column by common names
label_candidates = ["label", "label_str", "attack_cat", "class", "target", "y"]
lbl_col = next((c for c in label_candidates if c in DF.columns), None)
assert lbl_col is not None, f"No label column found. Columns: {list(DF.columns)[:30]}"

# Map to binary (benign=0 else 1) in a robust way
s = DF[lbl_col]
benign_tokens = {"benign", "normal", "background", "bg", "clean"}
if pd.api.types.is_numeric_dtype(s) and set(pd.unique(s.dropna())) <= {0, 1}:
    y = s.astype(int).values
else:
    y = (~s.astype("string").str.strip().str.lower().isin(benign_tokens)).astype(int).values

DF = DF.copy()
DF["label"] = y
X_full = DF.drop(columns=["label"])

print("Dataset:", DF.shape, "| Positive rate:", float(np.mean(y)))


Dataset: (79881, 1505) | Positive rate: 0.7371089495624742


## 4. Preprocessing and Train/Test Split
- Cast `object` → `category` for LGBM.
- Align category vocab between train and test.
- Stratified split to preserve class balance.


In [4]:
def prep_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Cast objects to categorical; bool to int8; leave numerics untouched."""
    Z = df.copy()
    for c in Z.columns:
        if c == "label":
            continue
        if Z[c].dtype == bool:
            Z[c] = Z[c].astype(np.int8)
        elif Z[c].dtype == object:
            Z[c] = Z[c].astype("category")
    return Z

Z = prep_frame(DF)
X = Z.drop(columns=["label"])
y = Z["label"].astype(int).values

# Stratified split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# Align category vocab across splits
cat_cols = [c for c in X_train.columns if str(X_train[c].dtype) == "category"]
for c in cat_cols:
    cats = pd.Index(pd.concat([X_train[c].astype("string"), X_test[c].astype("string")]).unique())
    X_train[c] = pd.Categorical(X_train[c].astype("string"), categories=cats)
    X_test[c]  = pd.Categorical(X_test[c].astype("string"),  categories=cats)

print(f"Train: {X_train.shape} | Test: {X_test.shape} | #categorical: {len(cat_cols)}")


Train: (59910, 1504) | Test: (19971, 1504) | #categorical: 1


## 5. Baseline: LightGBM
A strong tabular baseline. Uses `class_weight='balanced'` and categorical features directly.


In [5]:
from lightgbm import LGBMClassifier

# Train baseline
baseline = LGBMClassifier(
    objective="binary", n_estimators=400, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
)
fit_kwargs = {"categorical_feature": cat_cols} if cat_cols else {}
baseline.fit(X_train, y_train, **fit_kwargs)

# Evaluate
p_base = baseline.predict_proba(X_test)[:, 1]
ap_base = average_precision_score(y_test, p_base)
f1_base, tau_base = sweep_f1(y_test, p_base)
tpr_b, tnr_b = tpr_tnr_at_tau(y_test, p_base, tau_base)

# Stage
save_stage("baseline", {
    "timestamp": now_utc(), "stage": "baseline", "model_name": "LightGBM_Baseline",
    "dataset": DATA_PATH, "seed": RANDOM_STATE,
    "metrics": {"AUPRC": float(ap_base), "F1": float(f1_base), "tau": float(tau_base),
                "TPR": float(tpr_b), "TNR": float(tnr_b)}
}, model=baseline)

print(f"Baseline | AUPRC={ap_base:.4f} | F1@τ={f1_base:.4f} | τ={tau_base:.2f} | TPR={tpr_b:.3f} | TNR={tnr_b:.3f}")


Baseline | AUPRC=0.9998 | F1@τ=0.9997 | τ=0.05 | TPR=1.000 | TNR=0.998


/var/folders/sh/njlwqz496112jgym_0_p82h80000gn/T/ipykernel_59075/713894992.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat() + "Z"


## 6. Hyperparameter Optimisation (HPO)
We optimise **PR-AUC** via stratified CV using:
- **6.1** Manual Grid (parallel + resumable)
- **6.2** RandomizedSearchCV (broad)
- **6.3** BayesSearchCV (BO + checkpoint)
- **6.4** Enhanced BO (ask–tell, warm start, batch EI, patience, resume)


### 6.1 Manual Grid (parallel + resumable)
- Small, interpretable grid.
- Robust resume logic: safely reads older/partial CSVs.


In [ ]:
from joblib import Parallel, delayed
import pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier

mg_dir = stage_dir("manual_grid")
mg_csv = mg_dir / "results.csv"

# Define grid
grid = {
    "num_leaves": [31, 63, 127],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "min_child_samples": [20, 50],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def cv_ap(params: dict) -> dict:
    """Return dict(params, AP_cv) by 3-fold stratified CV Average Precision."""
    clf = LGBMClassifier(
        objective="binary", n_estimators=500, random_state=RANDOM_STATE,
        n_jobs=-1, class_weight="balanced", verbosity=-1, **params
    )
    aps = []
    for tr, va in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[tr], X_train.iloc[va]
        y_tr, y_va = y_train[tr], y_train[va]
        fit_kwargs = {"categorical_feature": cat_cols} if cat_cols else {}
        clf.fit(X_tr, y_tr, **fit_kwargs)
        p = clf.predict_proba(X_va)[:, 1]
        aps.append(average_precision_score(y_va, p))
    return {**params, "AP_cv": float(np.mean(aps))}

# Load previous results (handle partial/old CSVs gracefully)
if mg_csv.exists():
    prev_raw = pd.read_csv(mg_csv)
    missing = [k for k in grid.keys() if k not in prev_raw.columns]
    prev_for_dedupe = pd.DataFrame(columns=list(grid.keys()) + ["AP_cv"]) if missing else prev_raw.copy()
else:
    prev_raw = pd.DataFrame()
    prev_for_dedupe = pd.DataFrame(columns=list(grid.keys()) + ["AP_cv"])

# Deduplicate by parameter tuple
def row_key(row: pd.Series) -> tuple:
    return tuple((k, row[k]) for k in grid.keys())

done = set()
if not prev_for_dedupe.empty:
    for _, r in prev_for_dedupe.iterrows():
        try:
            done.add(row_key(r))
        except KeyError:
            continue

# Generate todo list
todo = []
for nl in grid["num_leaves"]:
    for md in grid["max_depth"]:
        for lr in grid["learning_rate"]:
            for mcs in grid["min_child_samples"]:
                for ss in grid["subsample"]:
                    for cs in grid["colsample_bytree"]:
                        params = {"num_leaves": nl, "max_depth": md, "learning_rate": lr,
                                  "min_child_samples": mcs, "subsample": ss, "colsample_bytree": cs}
                        if tuple(params.items()) not in done:
                            todo.append(params)

# Evaluate pending points in parallel
new_rows = []
if todo:
    n_jobs = min(N_THREADS, len(todo))
    new_rows = Parallel(n_jobs=n_jobs)(delayed(cv_ap)(p) for p in todo)

# Merge new + previous; persist for future resumes
mg_df = (pd.concat([prev_raw, pd.DataFrame(new_rows)], ignore_index=True)
         if new_rows else prev_raw.copy())
mg_df.to_csv(mg_csv, index=False)

# Select best among rows with all grid keys and AP_cv
eligible_cols = list(grid.keys()) + ["AP_cv"]
eligible = (mg_df.dropna(subset=["AP_cv"])[eligible_cols]
            if set(eligible_cols).issubset(mg_df.columns) else pd.DataFrame())
if eligible.empty:
    raise RuntimeError("Manual Grid has no complete evaluations yet; re-run this cell.")

best = eligible.sort_values("AP_cv", ascending=False).iloc[0].to_dict()
best_params = {k: best[k] for k in grid.keys()}

# Train best on full train, evaluate on test
lgb_mg = LGBMClassifier(
    objective="binary", n_estimators=500, random_state=RANDOM_STATE,
    n_jobs=-1, class_weight="balanced", verbosity=-1, **best_params
)
fit_kwargs = {"categorical_feature": cat_cols} if cat_cols else {}
lgb_mg.fit(X_train, y_train, **fit_kwargs)

p = lgb_mg.predict_proba(X_test)[:, 1]
ap = average_precision_score(y_test, p)
f1b, tau = sweep_f1(y_test, p)
tpr, tnr = tpr_tnr_at_tau(y_test, p, tau)

save_stage("manual_grid", {
    "timestamp": now_utc(), "stage": "manual_grid", "model_name": "LightGBM_ManualGrid",
    "dataset": DATA_PATH, "seed": RANDOM_STATE,
    "metrics": {"AUPRC": float(ap), "F1": float(f1b), "tau": float(tau), "TPR": float(tpr), "TNR": float(tnr)},
    "params": best_params
}, results=mg_df, model=lgb_mg)

print(f"Manual Grid | best CV AP={best['AP_cv']:.4f} | Test AP={ap:.4f} | F1@τ={f1b:.4f} (τ={tau:.2f})")


### 6.2 Randomized Search (broad)
Covers a wider region efficiently; parallel; refits best on full train.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, loguniform
from lightgbm import LGBMClassifier

param_dist = {
    "num_leaves": randint(16, 256),
    "max_depth": randint(2, 16),
    "min_child_samples": randint(10, 200),
    "subsample": loguniform(0.6, 1.0),
    "colsample_bytree": loguniform(0.6, 1.0),
    "learning_rate": loguniform(1e-3, 2e-1),
    "reg_lambda": loguniform(1e-3, 10.0),
}

base = LGBMClassifier(
    objective="binary", n_estimators=500, random_state=RANDOM_STATE,
    n_jobs=-1, class_weight="balanced", verbosity=-1
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

rs = RandomizedSearchCV(
    base, param_dist, n_iter=40, scoring="average_precision",
    cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1, refit=True
)

fit_kwargs = {"categorical_feature": cat_cols} if cat_cols else {}
rs.fit(X_train, y_train, **fit_kwargs)

best_rs = rs.best_estimator_
p = best_rs.predict_proba(X_test)[:, 1]
ap = average_precision_score(y_test, p)
f1b, tau = sweep_f1(y_test, p)
tpr, tnr = tpr_tnr_at_tau(y_test, p, tau)

rs_df = pd.DataFrame(rs.cv_results_)
save_stage("random_search", {
    "timestamp": now_utc(), "stage": "random_search", "model_name": "LightGBM_RS",
    "dataset": DATA_PATH, "seed": RANDOM_STATE,
    "metrics": {"AUPRC": float(ap), "F1": float(f1b), "tau": float(tau), "TPR": float(tpr), "TNR": float(tnr)},
    "best_params": rs.best_params_
}, results=rs_df, model=best_rs)

print(f"Randomized Search | best CV AP={rs.best_score_:.4f} | Test AP={ap:.4f} | F1@τ={f1b:.4f} (τ={tau:.2f})")


### 6.3 Bayesian Optimisation (BayesSearchCV) with Checkpoint
Sample-efficient BO over defined spaces; safe checkpointing for resume.


In [ ]:
from skopt import BayesSearchCV
from skopt.space import Integer, Real
from skopt.callbacks import CheckpointSaver
from lightgbm import LGBMClassifier

bo_dir = stage_dir("bo_lgb")
ckpt = bo_dir / "skopt_ckpt.pkl"

lgb = LGBMClassifier(
    objective="binary", n_estimators=500, random_state=RANDOM_STATE,
    n_jobs=-1, class_weight="balanced", verbosity=-1
)

search_spaces = {
    "num_leaves": Integer(31, 255),
    "max_depth": Integer(2, 16),
    "min_child_samples": Integer(10, 200),
    "subsample": Real(0.6, 1.0),
    "colsample_bytree": Real(0.6, 1.0),
    "learning_rate": Real(1e-3, 2e-1, prior="log-uniform"),
    "reg_lambda": Real(1e-3, 10.0, prior="log-uniform"),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
callbacks = [CheckpointSaver(str(ckpt), compress=9, store_objective=False)]

opt = BayesSearchCV(
    estimator=lgb, search_spaces=search_spaces, n_iter=40,
    scoring="average_precision", cv=cv, n_jobs=-1,
    random_state=RANDOM_STATE, verbose=1, refit=True
)

fit_kwargs = {"categorical_feature": cat_cols} if cat_cols else {}
opt.fit(X_train, y_train, callback=callbacks, **fit_kwargs)

lgb_bo = opt.best_estimator_
p = lgb_bo.predict_proba(X_test)[:, 1]
ap = average_precision_score(y_test, p)
f1b, tau = sweep_f1(y_test, p)
tpr, tnr = tpr_tnr_at_tau(y_test, p, tau)

save_stage("bo_lgb", {
    "timestamp": now_utc(), "stage": "bo_lgb", "model_name": "LightGBM_BO",
    "dataset": DATA_PATH, "seed": RANDOM_STATE,
    "metrics": {"AUPRC": float(ap), "F1": float(f1b), "tau": float(tau), "TPR": float(tpr), "TNR": float(tnr)},
    "best_params": opt.best_params_
}, model=lgb_bo)

print(f"BO | best CV AP={opt.best_score_:.4f} | Test AP={ap:.4f} | F1@τ={f1b:.4f} (τ={tau:.2f})")


### 6.4 Enhanced BO (ask–tell, warm start, batch EI, patience, resume)
- Warm-start from prior stages; batch parallel EI; patience early-stopping.
- Full resume via optimizer pickle + CSV history.


In [ ]:
from skopt import Optimizer
from skopt.space import Integer, Real
from joblib import Parallel, delayed
from lightgbm import LGBMClassifier

ben_dir = stage_dir("bo_enhanced")
opt_pkl = ben_dir / "optimizer.pkl"
res_csv = ben_dir / "results.csv"

space = [
    Integer(31, 255, name="num_leaves"),
    Integer(2, 16, name="max_depth"),
    Integer(10, 200, name="min_child_samples"),
    Real(0.6, 1.0, name="subsample"),
    Real(0.6, 1.0, name="colsample_bytree"),
    Real(1e-3, 2e-1, prior="log-uniform", name="learning_rate"),
    Real(1e-3, 10.0, prior="log-uniform", name="reg_lambda"),
]

# Warm-start from prior bests
X0, y0 = [], []
for stg in ["manual_grid", "random_search", "bo_lgb"]:
    m = load_stage(stg)
    if not m: continue
    p = m.get("best_params") or m.get("params")
    if not p: continue
    x = [int(p.get("num_leaves",63)), int(p.get("max_depth",6)), int(p.get("min_child_samples",20)),
         float(p.get("subsample",0.8)), float(p.get("colsample_bytree",0.8)),
         float(p.get("learning_rate",0.1)), float(p.get("reg_lambda",1.0))]
    ap = m.get("metrics",{}).get("AUPRC")
    if ap is not None:
        X0.append(x); y0.append(-float(ap))  # optimizer minimizes

# Build/resume optimizer
if opt_pkl.exists():
    opt = joblib.load(opt_pkl)
else:
    opt = Optimizer(dimensions=space, base_estimator="GP", acq_func="EI", random_state=RANDOM_STATE)
    if X0:
        opt.tell(X0, y0)

def cv_ap_eval(params: dict) -> float:
    """Return mean CV AP for given params (used by ask–tell loop)."""
    clf = LGBMClassifier(
        objective="binary", n_estimators=500, random_state=RANDOM_STATE,
        n_jobs=-1, class_weight="balanced", verbosity=-1, **params
    )
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    aps = []
    for tr, va in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[tr], X_train.iloc[va]
        y_tr, y_va = y_train[tr], y_train[va]
        fit_kwargs = {"categorical_feature": cat_cols} if cat_cols else {}
        clf.fit(X_tr, y_tr, **fit_kwargs)
        p = clf.predict_proba(X_va)[:, 1]
        aps.append(average_precision_score(y_va, p))
    return float(np.mean(aps))

def vec_to_params(v):
    return {"num_leaves": int(v[0]), "max_depth": int(v[1]), "min_child_samples": int(v[2]),
            "subsample": float(v[3]), "colsample_bytree": float(v[4]),
            "learning_rate": float(v[5]), "reg_lambda": float(v[6])}

# History table (resume)
hist = pd.read_csv(res_csv) if res_csv.exists() else pd.DataFrame(columns=[
    "num_leaves","max_depth","min_child_samples","subsample","colsample_bytree","learning_rate","reg_lambda","AP_cv"
])

N_ITERS, BATCH, PATIENCE = 30, min(4, N_THREADS), 6
best_cv, no_imp = (hist["AP_cv"].max() if not hist.empty else -np.inf), 0

for step in range(N_ITERS):
    Xb = opt.ask(n_points=BATCH)
    Pb = [vec_to_params(v) for v in Xb]
    scores = Parallel(n_jobs=BATCH)(delayed(cv_ap_eval)(p) for p in Pb)
    opt.tell(Xb, [-s for s in scores])

    rows = [dict(**p, AP_cv=s) for p, s in zip(Pb, scores)]
    hist = pd.concat([hist, pd.DataFrame(rows)], ignore_index=True)
    hist.to_csv(res_csv, index=False)
    joblib.dump(opt, opt_pkl)

    b = float(max(scores))
    if b > best_cv + 1e-6:
        best_cv, no_imp = b, 0
    else:
        no_imp += 1
    print(f"[Enhanced BO] step {step+1}/{N_ITERS} | best_cv_AP={best_cv:.4f} | patience {no_imp}/{PATIENCE}")
    if no_imp >= PATIENCE:
        print("Early stopping (patience)."); break

# Train best on full train & stage
if not hist.empty:
    best_row = hist.sort_values("AP_cv", ascending=False).iloc[0].to_dict()
    best_params = {k: best_row[k] for k in ["num_leaves","max_depth","min_child_samples","subsample","colsample_bytree","learning_rate","reg_lambda"]}
    lgb_enh = LGBMClassifier(
        objective="binary", n_estimators=500, random_state=RANDOM_STATE,
        n_jobs=-1, class_weight="balanced", verbosity=-1, **best_params
    )
    fit_kwargs = {"categorical_feature": cat_cols} if cat_cols else {}
    lgb_enh.fit(X_train, y_train, **fit_kwargs)
    p = lgb_enh.predict_proba(X_test)[:, 1]
    ap = average_precision_score(y_test, p)
    f1b, tau = sweep_f1(y_test, p)
    tpr, tnr = tpr_tnr_at_tau(y_test, p, tau)

    save_stage("bo_enhanced", {
        "timestamp": now_utc(), "stage": "bo_enhanced", "model_name": "LightGBM_BO_Enhanced",
        "dataset": DATA_PATH, "seed": RANDOM_STATE,
        "metrics": {"AUPRC": float(ap), "F1": float(f1b), "tau": float(tau), "TPR": float(tpr), "TNR": float(tnr)},
        "params": best_params
    }, results=hist, model=lgb_enh)

    print(f"Enhanced BO | best CV AP={best_row['AP_cv']:.4f} | Test AP={ap:.4f} | F1@τ={f1b:.4f} (τ={tau:.2f})")
else:
    print("Enhanced BO produced no evaluations.")


## 7. 1D-CNN Baseline (optional)
Treat tabular features as a 1D sequence: Conv1D layers + global pooling.
Saved predictions integrate into ensembles/plots. Skips if TF unavailable.


In [ ]:
# Minimal 1D-CNN with graceful skip if TensorFlow absent
try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Input, Conv1D, GlobalMaxPooling1D, Dense, Dropout
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    try:
        from tensorflow.keras.callbacks import BackupAndRestore
        _has_backup = True
    except Exception:
        _has_backup = False
    TF_OK = True
except Exception:
    TF_OK = False

def to_seq(X_train: pd.DataFrame, X_test: pd.DataFrame):
    """Convert DataFrame to numeric tensor [N, F, 1] (categories->codes, bool->int)."""
    A, B = X_train.copy(), X_test.copy()
    for c in A.columns:
        if str(A[c].dtype) == "category":
            A[c] = A[c].cat.codes; B[c] = B[c].cat.codes
        elif A[c].dtype == bool:
            A[c] = A[c].astype(np.int8); B[c] = B[c].astype(np.int8)
        elif A[c].dtype == object:
            A[c] = A[c].astype("category").cat.codes
            B[c] = B[c].astype("category").cat.codes
    A = A.fillna(0).astype(np.float32).values
    B = B.fillna(0).astype(np.float32).values
    return A.reshape((A.shape[0], A.shape[1], 1)), B.reshape((B.shape[0], B.shape[1], 1))

if TF_OK:
    Xtr3, Xte3 = to_seq(X_train, X_test)
    L = Xtr3.shape[1]
    model = Sequential([
        Input(shape=(L,1)),
        Conv1D(32, 3, activation="relu", padding="same"),
        Conv1D(32, 3, activation="relu", padding="same"),
        GlobalMaxPooling1D(),
        Dropout(0.2),
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=Adam(1e-3), loss="binary_crossentropy",
                  metrics=[tf.keras.metrics.AUC(curve="PR", name="AUPRC")])
    callbacks=[EarlyStopping(monitor="val_AUPRC", mode="max", patience=5, restore_best_weights=True),
               ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)]
    cnn_stage = stage_dir("cnn1d")
    if _has_backup:
        callbacks.append(BackupAndRestore(backup_dir=str(cnn_stage/"tf_backup")))
    hist = model.fit(
        Xtr3, y_train, validation_split=0.2, epochs=30, batch_size=256,
        callbacks=callbacks, verbose=1
    )
    p = model.predict(Xte3, batch_size=1024, verbose=0).ravel()
    ap = average_precision_score(y_test, p)
    f1b, tau = sweep_f1(y_test, p)
    tpr, tnr = tpr_tnr_at_tau(y_test, p, tau)
    np.save(cnn_stage/"preds.npy", p)
    model.save(cnn_stage/"model.keras", include_optimizer=False)
    save_stage("cnn1d", {
        "timestamp": now_utc(), "stage": "cnn1d", "model_name": "CNN1D_Minimal",
        "dataset": DATA_PATH, "seed": RANDOM_STATE,
        "metrics": {"AUPRC": float(ap), "F1": float(f1b), "tau": float(tau), "TPR": float(tpr), "TNR": float(tnr)}
    })
    print(f"CNN1D | AUPRC={ap:.4f} | F1@τ={f1b:.4f} | τ={tau:.2f} | TPR={tpr:.3f} | TNR={tnr:.3f}")
else:
    print("TensorFlow not available — skipping CNN1D.")


## 8. Ensembles (RF Bagging, Soft Voting, Blended Stacking)
Combine complementary models to reduce variance and improve AUPRC.  
We avoid test-tuned weights; soft voting uses **uniform** weights by default.


### 8A. Random Forest Bagging

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500, class_weight="balanced_subsample",
    n_jobs=-1, random_state=RANDOM_STATE
).fit(X_train, y_train)

p = rf.predict_proba(X_test)[:, 1]
ap = average_precision_score(y_test, p)
f1b, tau = sweep_f1(y_test, p)
tpr, tnr = tpr_tnr_at_tau(y_test, p, tau)

save_stage("ensemble_rf", {
    "timestamp": now_utc(), "stage": "ensemble_rf", "model_name": "RandomForest_Bagging",
    "dataset": DATA_PATH, "seed": RANDOM_STATE,
    "metrics": {"AUPRC": float(ap), "F1": float(f1b), "tau": float(tau), "TPR": float(tpr), "TNR": float(tnr)},
    "params": rf.get_params()
}, model=rf)

print(f"RF Bagging | AUPRC={ap:.4f} | F1@τ={f1b:.4f} | τ={tau:.2f}")


### 8B. Soft Voting (uniform weights; safe)

In [ ]:
from pathlib import Path

def load_preds_for(stage_name: str):
    """Try to load test-set probabilities for a staged model."""
    p = Path("staging")/stage_name/"model.joblib"
    if p.exists():
        mdl = joblib.load(p)
        try:
            return mdl.predict_proba(X_test)[:, 1]
        except Exception:
            return None
    if stage_name == "cnn1d":
        n = Path("staging")/"cnn1d"/"preds.npy"
        if n.exists():
            arr = np.load(n)
            return arr if len(arr) == len(y_test) else None
    return None

members = ["bo_enhanced", "bo_lgb", "random_search", "manual_grid", "ensemble_rf", "baseline", "cnn1d"]
preds = {m: load_preds_for(m) for m in members}
preds = {k: v for k, v in preds.items() if v is not None}

if len(preds) >= 2:
    W = np.ones(len(preds)) / len(preds)               # uniform weights (no leakage)
    P = np.stack(list(preds.values()), axis=1)
    p_vote = (P * W).sum(axis=1)

    ap = average_precision_score(y_test, p_vote)
    f1b, tau = sweep_f1(y_test, p_vote)
    tpr, tnr = tpr_tnr_at_tau(y_test, p_vote, tau)

    save_stage("ensemble_soft", {
        "timestamp": now_utc(), "stage": "ensemble_soft", "model_name": "Ensemble_SoftVoting",
        "dataset": DATA_PATH, "seed": RANDOM_STATE,
        "members": list(preds.keys()), "weights": {m: float(1/len(preds)) for m in preds.keys()},
        "metrics": {"AUPRC": float(ap), "F1": float(f1b), "tau": float(tau), "TPR": float(tpr), "TNR": float(tnr)}
    })
    print(f"Soft Voting | members={list(preds.keys())} | AUPRC={ap:.4f} | F1@τ={f1b:.4f} | τ={tau:.2f}")
else:
    print("Soft voting requires ≥2 staged predictors.")


### 8C. Blended Stacking (train base on A, meta on B)

In [ ]:
from sklearn.linear_model import LogisticRegression

# Collect best LightGBM params from staged runs
specs = []
for s in ["bo_enhanced", "bo_lgb", "random_search", "manual_grid"]:
    m = load_stage(s)
    if not m: continue
    p = m.get("best_params") or m.get("params")
    if p: specs.append((s, p))

if not specs:
    print("No LightGBM staged params for stacking.")
else:
    XA, XB, yA, yB = train_test_split(X_train, y_train, test_size=0.2,
                                      random_state=RANDOM_STATE, stratify=y_train)
    ZB, ZT, names = [], [], []
    for name, params in specs:
        clf = LGBMClassifier(
            objective="binary", n_estimators=500, random_state=RANDOM_STATE,
            n_jobs=-1, class_weight="balanced", verbosity=-1, **params
        )
        fit_kwargs = {"categorical_feature": cat_cols} if cat_cols else {}
        clf.fit(XA, yA, **fit_kwargs)
        ZB.append(clf.predict_proba(XB)[:, 1])      # level-1 features for meta-train
        ZT.append(clf.predict_proba(X_test)[:, 1])  # level-1 features for meta-test
        names.append(name)

    ZB = np.stack(ZB, axis=1); ZT = np.stack(ZT, axis=1)
    meta = LogisticRegression(class_weight="balanced", max_iter=300, random_state=RANDOM_STATE)
    meta.fit(ZB, yB)
    p_stack = meta.predict_proba(ZT)[:, 1]

    ap = average_precision_score(y_test, p_stack)
    f1b, tau = sweep_f1(y_test, p_stack)
    tpr, tnr = tpr_tnr_at_tau(y_test, p_stack, tau)

    save_stage("ensemble_stack", {
        "timestamp": now_utc(), "stage": "ensemble_stack", "model_name": "Ensemble_BlendedStack",
        "dataset": DATA_PATH, "seed": RANDOM_STATE, "members": names, "meta": "LogisticRegression",
        "metrics": {"AUPRC": float(ap), "F1": float(f1b), "tau": float(tau), "TPR": float(tpr), "TNR": float(tnr)}
    })
    print(f"Stacking | AUPRC={ap:.4f} | F1@τ={f1b:.4f} | τ={tau:.2f} | members={names}")


## 9. 1D-CNN Baseline (Deep Learning Benchmark)

As a baseline comparator, we build a **1D Convolutional Neural Network (1D-CNN)** using Keras/TensorFlow.  
This allows us to benchmark our **Bayesian-Optimised LightGBM** results against a compact deep learning model, consistent with the IDS literature.

- **Why 1D-CNN?**  
  - IDS data is tabular but often treated as sequential feature vectors.  
  - 1D convolutions can capture local correlations across features.  
  - Provides a non-tree, deep-learning baseline for fairer comparisons.

We will:
1. Preprocess data into numpy arrays.
2. Define a lightweight 1D-CNN.
3. Train with early stopping.
4. Evaluate on test set with PR-AUC and F1.


In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
# Convert features to numpy arrays for Keras
X_train_np = X_train.values.astype("float32")
X_test_np  = X_test.values.astype("float32")

# Add channel dimension for Conv1D: (samples, timesteps, channels)
X_train_cnn = X_train_np[..., np.newaxis]
X_test_cnn  = X_test_np[..., np.newaxis]

y_train_np = y_train.values
y_test_np  = y_test.values


In [ ]:
def build_1dcnn(input_shape):
    model = Sequential([
        Conv1D(filters=32, kernel_size=3, activation="relu", input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        Conv1D(filters=64, kernel_size=3, activation="relu"),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["AUC", "Precision", "Recall"])
    return model

cnn_model = build_1dcnn((X_train_cnn.shape[1], 1))
cnn_model.summary()


In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history = cnn_model.fit(
    X_train_cnn, y_train_np,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=2
)


In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, f1_score

y_pred_prob = cnn_model.predict(X_test_cnn).ravel()
ap_score = average_precision_score(y_test_np, y_pred_prob)

# Choose threshold for max-F1
prec, rec, thr = precision_recall_curve(y_test_np, y_pred_prob)
f1_scores = 2*prec*rec / (prec+rec+1e-9)
best_idx = np.argmax(f1_scores)
best_thr, best_f1 = thr[best_idx], f1_scores[best_idx]

print(f"1D-CNN PR-AUC: {ap_score:.4f}")
print(f"Best F1: {best_f1:.4f} at threshold {best_thr:.4f}")


## 9. Analysis & Visual Justification
- Aggregate metrics table.
- Enhanced BO convergence (best-so-far CV AP).
- Hyperparameter response plots (param vs CV AP).
- PR curves (Champion vs others).
- F1 vs τ curves.
- Feature importance (LightGBM champion).


### 9A. Aggregate Metrics Table

In [ ]:
stages = ["baseline","manual_grid","random_search","bo_lgb","bo_enhanced",
          "ensemble_rf","ensemble_soft","ensemble_stack","cnn1d"]
rows = []
for s in stages:
    m = load_stage(s)
    if m and "metrics" in m:
        rows.append({"stage": s, **m["metrics"]})
metrics_df = pd.DataFrame(rows)
display(metrics_df.sort_values(["AUPRC","F1"], ascending=[False, False]).reset_index(drop=True)
        if not metrics_df.empty else "No staged metrics yet.")


### 9B. Enhanced BO convergence (best-so-far CV AP)

In [ ]:
import matplotlib.pyplot as plt
ben_hist = stage_dir("bo_enhanced") / "results.csv"
plt.figure()
if ben_hist.exists():
    h = pd.read_csv(ben_hist)
    if not h.empty and "AP_cv" in h.columns:
        best_so_far = h["AP_cv"].cummax().values
        plt.plot(range(1, len(best_so_far)+1), best_so_far, marker='o', linewidth=1)
        plt.xlabel("Enhanced BO evaluations"); plt.ylabel("Best CV AP so far")
        plt.title("Enhanced BO Convergence"); plt.grid(True, linewidth=0.3)
    else:
        plt.text(0.5, 0.5, "No AP_cv history.", ha="center")
else:
    plt.text(0.5, 0.5, "No bo_enhanced/results.csv.", ha="center")
plt.show()


### 9C. Hyperparameter response plots (param vs CV AP)

In [ ]:
hpath = stage_dir("bo_enhanced") / "results.csv"
if hpath.exists():
    h = pd.read_csv(hpath)
    hp_cols = ["num_leaves","max_depth","min_child_samples","subsample",
               "colsample_bytree","learning_rate","reg_lambda"]
    for c in hp_cols:
        if c in h.columns:
            plt.figure()
            plt.scatter(h[c].values, h["AP_cv"].values, s=12)
            plt.xlabel(c); plt.ylabel("CV AP"); plt.title(f"{c} vs CV AP")
            plt.grid(True, linewidth=0.3); plt.show()
else:
    print("No enhanced BO history.")


### 9D. PR curves (Champion vs others)

In [ ]:
def pr_arrays(y_true, p):
    P, R, T = precision_recall_curve(y_true, p); ap = average_precision_score(y_true, p)
    return P, R, ap

# Collect predictions
curves = {"baseline": p_base}
# models from staging
for s in ["bo_enhanced","bo_lgb","random_search","manual_grid","ensemble_rf","ensemble_soft","ensemble_stack"]:
    mp = stage_dir(s) / "model.joblib"
    if mp.exists():
        mdl = joblib.load(mp)
        try:
            curves[s] = mdl.predict_proba(X_test)[:, 1]
        except Exception:
            pass
# CNN preds
pn = stage_dir("cnn1d") / "preds.npy"
if pn.exists():
    curves["cnn1d"] = np.load(pn)

plt.figure()
for name, pr in curves.items():
    P, R, AP = pr_arrays(y_test, pr)
    plt.plot(R, P, linewidth=1, label=f"{name} (AP={AP:.3f})")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision–Recall Curves")
plt.grid(True, linewidth=0.3); plt.legend(); plt.show()


### 9E. F1 vs τ curves

In [ ]:
taus = np.linspace(0.01, 0.99, 50)
def f1_curve(y_true, p, ts): return np.array([f1_score(y_true, (p>=t).astype(int), zero_division=0) for t in ts])

# Champion by staged AUPRC→F1
allm = {}
for s in ["manual_grid","random_search","bo_lgb","bo_enhanced","ensemble_rf","ensemble_soft","ensemble_stack","baseline","cnn1d"]:
    m = load_stage(s)
    if m and "metrics" in m:
        allm[s] = m["metrics"]
champ = None
if allm:
    dfm = pd.DataFrame(allm).T
    champ = dfm.sort_values(["AUPRC","F1"], ascending=[False, False]).index[0]

curves_tau = {}
# ensure champion preds available
if champ:
    if champ == "baseline":
        curves_tau["_champ_"] = p_base
    else:
        mp = stage_dir(champ) / "model.joblib"
        if mp.exists():
            mdl = joblib.load(mp)
            try:
                curves_tau["_champ_"] = mdl.predict_proba(X_test)[:, 1]
            except Exception:
                pass
# add CNN if present
pn = stage_dir("cnn1d") / "preds.npy"
if pn.exists():
    curves_tau["cnn1d"] = np.load(pn)

plt.figure()
for name, p in curves_tau.items():
    vals = f1_curve(y_test, p, taus)
    plt.plot(taus, vals, linewidth=1, label=name)
plt.xlabel("τ"); plt.ylabel("F1"); plt.title("F1 vs τ")
plt.grid(True, linewidth=0.3); plt.legend(); plt.show()


### 9F. Feature importance (LightGBM champion)

In [ ]:
champ_model, champ_stage = None, None
for s in ["bo_enhanced","bo_lgb","random_search","manual_grid","baseline"]:
    mp = stage_dir(s) / "model.joblib"
    if mp.exists():
        mdl = joblib.load(mp)
        if hasattr(mdl, "booster_"):   # LightGBM model
            champ_model, champ_stage = mdl, s
            break

if champ_model is not None:
    booster = champ_model.booster_
    try:
        gains = booster.feature_importance(importance_type="gain")
        names = booster.feature_name()
        imp_df = pd.DataFrame({"feature": names, "gain": gains}).sort_values("gain", ascending=False).head(20)
        plt.figure(figsize=(8,6))
        plt.barh(range(len(imp_df)), imp_df["gain"].values)
        plt.yticks(range(len(imp_df)), imp_df["feature"].values)
        plt.gca().invert_yaxis()
        plt.xlabel("Gain"); plt.title(f"Top-20 Feature Importances ({champ_stage})")
        plt.tight_layout(); plt.show()
    except Exception as e:
        print("Importance extraction failed:", e)
else:
    print("No LightGBM champion model available for importance.")


## 10. Auto-Wire Docs (Champion + Optional OOF)
Update `model_card.md` and `README.md` between markers `<!--METRICS_START-->...<!--METRICS_END-->` with:
- **Champion** (test) metrics,
- **Model_Evaluation_Tools (OOF)** metrics if that notebook has been run.


In [ ]:
def load_metrics_table(include=("manual_grid","random_search","bo_lgb","bo_enhanced",
                                "ensemble_rf","ensemble_soft","ensemble_stack","cnn1d","baseline")):
    M = {}
    for s in include:
        m = load_stage(s)
        if m and "metrics" in m:
            M[s] = m["metrics"]
    return pd.DataFrame(M).T if M else pd.DataFrame()

dfm = load_metrics_table()
if not dfm.empty:
    champ_name = dfm.sort_values(["AUPRC","F1"], ascending=[False, False]).index[0]
    met = load_stage(champ_name)["metrics"]
    champ_block = (
        "### Champion (staging)\n"
        f"- **Stage:** {champ_name}\n"
        f"- **AUPRC:** {met.get('AUPRC', float('nan')):.4f}\n"
        f"- **F1:** {met.get('F1', float('nan')):.4f}\n"
        f"- **τ:** {met.get('tau', float('nan')):.2f}\n"
        f"- **TPR/TNR @ τ:** {met.get('TPR', float('nan')):.3f} / {met.get('TNR', float('nan')):.3f}\n"
    )
else:
    champ_block = "### Champion (staging)\n- No champion metrics available.\n"

# Optional OOF block from Model_Evaluation_Tools
eval_manifest = Path("staging/eval_cv/manifest.json")
if eval_manifest.exists():
    try:
        em = json.loads(eval_manifest.read_text()); mm = em.get("metrics", {})
        eval_block = (
            "### Cross-Validation (OOF) — Model_Evaluation_Tools\n"
            f"- **AUPRC (OOF):** {mm.get('AUPRC_OOF', float('nan')):.4f}\n"
            f"- **F1 (OOF):** {mm.get('F1_OOF', float('nan')):.4f}\n"
            f"- **τ (OOF):** {mm.get('tau', float('nan')):.2f}\n"
            f"- **TPR/TNR @ τ (OOF):** {mm.get('TPR', float('nan')):.3f} / {mm.get('TNR', float('nan')):.3f}\n"
        )
    except Exception:
        eval_block = "### Cross-Validation (OOF) — Model_Evaluation_Tools\n- Manifest present but unreadable.\n"
else:
    eval_block = "### Cross-Validation (OOF) — Model_Evaluation_Tools\n- Not found.\n"

block = "<!--METRICS_START-->\n" + champ_block + "\n" + eval_block + "<!--METRICS_END-->\n"

def inject(doc_path: str, block: str):
    """Insert/replace metrics block between markers in a doc file."""
    p = Path(doc_path)
    if not p.exists():
        return False
    txt = p.read_text()
    start, end = "<!--METRICS_START-->", "<!--METRICS_END-->"
    if start in txt and end in txt:
        pre, _, rest = txt.partition(start)
        _, _, post = rest.partition(end)
        txt = pre + block + post
    else:
        if not txt.endswith("\n"): txt += "\n"
        txt += "\n" + block
    p.write_text(txt); return True

updated = []
for doc in ["model_card.md", "README.md"]:
    if inject(doc, block):
        updated.append(doc)
print("Updated docs:", updated if updated else "None")


## 11. Version Log
- **v2.0** — Full rebuild: robust staging/resume; Grid/Random/BO/Enhanced BO; CNN; Ensembles; charts; auto-wire.
- **v1.4** — Prior BO with checkpoint & resume, initial analysis, doc auto-wire.
